In [1]:
import pandas as pd
import numpy as np

# Load your crop calendar dataset
df = pd.read_csv("crop_calendar_parsed.csv")
print(f"Original Dataset Shape: {df.shape}")

# Inspect head
df.head()

Original Dataset Shape: (6381, 17)


,Sl_No,State,District,District_Code,Crop,Season,Sowing_Period,Harvesting_Period,Sow_From_Day,Sow_From_Mon,Sow_To_Day,Sow_To_Mon,Harv_From_Day,Harv_From_Mon,Harv_To_Day,Harv_To_Mon,PDF_Page
0,1,Bihar,Arwal,240,Rice,Kharif,15th June - 15th Aug,15th Oct. – 30th Nov,15.0,6.0,15.0,8.0,15.0,10.0,30.0,11.0,1
1,2,Bihar,Arwal,240,Wheat,Rabi,15th Nov – 15th Dec,15th Mar. – 15th Apr,15.0,11.0,15.0,12.0,15.0,3.0,15.0,4.0,1
2,3,Bihar,Arwal,240,Barley,Rabi,15th Oct - 30th Nov,15th Mar- 15th Apr,15.0,10.0,30.0,11.0,15.0,3.0,15.0,4.0,1
3,4,Bihar,Arwal,240,Maize,Kharif,15th May – 15th June,1st -31st Sept,15.0,5.0,15.0,6.0,1.0,9.0,31.0,9.0,1
4,5,Bihar,Arwal,240,Maize,Rabi,15th Oct. – 15th Nov,1st – 30th Apr,15.0,10.0,15.0,11.0,1.0,4.0,30.0,4.0,1


In [2]:
# 1. Drop rows missing critical identifiers
df_clean = df.dropna(subset=['Crop', 'District']).copy()

# 2. Drop rows where BOTH visual date strings are missing 
# (Meaning we have no clue about sowing or harvesting timelines)
df_clean = df_clean.dropna(subset=['Sowing_Period', 'Harvesting_Period'], how='all')

# 3. Drop rows that have absolutely no numerical scheduling flags
# If a row lacks ALL starting metrics, it cannot be modeled or mapped to calendar dates
critical_numeric_fields = ['Sow_From_Day', 'Sow_From_Mon', 'Harv_From_Day', 'Harv_From_Mon']
df_clean = df_clean.dropna(subset=critical_numeric_fields, how='all')

print(f"Cleaned Dataset Shape after removing unusable entries: {df_clean.shape}")
print(f"Total Rows Removed: {len(df) - len(df_clean)}")

Cleaned Dataset Shape after removing unusable entries: (4949, 17)
Total Rows Removed: 1432


In [3]:
# Strip whitespace and normalize case rules for administrative fields
text_cols = ['State', 'District', 'Crop', 'Season']
for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip()

# Fill missing To_Day or To_Mon values with their corresponding From value as a fallback 
# (e.g., if it sows from Month 7 but doesn't state a To Month, assume it wraps within Month 7)
df_clean['Sow_To_Mon'] = df_clean['Sow_To_Mon'].fillna(df_clean['Sow_From_Mon'])
df_clean['Sow_To_Day'] = df_clean['Sow_To_Day'].fillna(df_clean['Sow_From_Day'])
df_clean['Harv_To_Mon'] = df_clean['Harv_To_Mon'].fillna(df_clean['Harv_From_Mon'])
df_clean['Harv_To_Day'] = df_clean['Harv_To_Day'].fillna(df_clean['Harv_From_Day'])

# Safely downcast calendar indexes to integers now that NaN values are mitigated
int_date_cols = ['Sow_From_Day', 'Sow_From_Mon', 'Sow_To_Day', 'Sow_To_Mon', 
                 'Harv_From_Day', 'Harv_From_Mon', 'Harv_To_Day', 'Harv_To_Mon']

for col in int_date_cols:
    df_clean[col] = df_clean[col].fillna(0).astype(int)

df_clean.head()

,Sl_No,State,District,District_Code,Crop,Season,Sowing_Period,Harvesting_Period,Sow_From_Day,Sow_From_Mon,Sow_To_Day,Sow_To_Mon,Harv_From_Day,Harv_From_Mon,Harv_To_Day,Harv_To_Mon,PDF_Page
0,1,Bihar,Arwal,240,Rice,Kharif,15th June - 15th Aug,15th Oct. – 30th Nov,15,6,15,8,15,10,30,11,1
1,2,Bihar,Arwal,240,Wheat,Rabi,15th Nov – 15th Dec,15th Mar. – 15th Apr,15,11,15,12,15,3,15,4,1
2,3,Bihar,Arwal,240,Barley,Rabi,15th Oct - 30th Nov,15th Mar- 15th Apr,15,10,30,11,15,3,15,4,1
3,4,Bihar,Arwal,240,Maize,Kharif,15th May – 15th June,1st -31st Sept,15,5,15,6,1,9,31,9,1
4,5,Bihar,Arwal,240,Maize,Rabi,15th Oct. – 15th Nov,1st – 30th Apr,15,10,15,11,1,4,30,4,1


In [4]:
missing_pct = (df_clean.isnull().sum() / len(df_clean)) * 100
missing_summary = pd.DataFrame({
    'Missing Count': df_clean.isnull().sum(),
    'Percentage': missing_pct.round(2)
}).sort_values(by='Percentage', ascending=False)

print("Remaining Missing Values Matrix:")
print(missing_summary)

Remaining Missing Values Matrix:
                   Missing Count  Percentage
Season                      1591       32.15
Harvesting_Period             46        0.93
District_Code                 42        0.85
Sl_No                          0        0.00
Sow_To_Day                     0        0.00
Harv_To_Mon                    0        0.00
Harv_To_Day                    0        0.00
Harv_From_Mon                  0        0.00
Harv_From_Day                  0        0.00
Sow_To_Mon                     0        0.00
Sow_From_Day                   0        0.00
Sow_From_Mon                   0        0.00
State                          0        0.00
Sowing_Period                  0        0.00
Crop                           0        0.00
District                       0        0.00
PDF_Page                       0        0.00


In [5]:
# 1. Define your strict target selector list
SELECTOR_CROPS = [
    "RICE", "WHEAT", "KHARIFSORGHUM", "RABISORGHUM", "SORGHUM", "PEARLMILLET", 
    "MAIZE", "FINGERMILLET", "BARLEY", "CHICKPEA", "PIGEONPEA", "MINORPULSES", 
    "GROUNDNUT", "SESAMUM", "RAPESEEDANDMUSTARD", "SAFFLOWER", "CASTOR", "LINSEED", 
    "SUNFLOWER", "SOYABEAN", "OILSEEDS", "SUGARCANE", "COTTON", "FRUITS", 
    "VEGETABLES", "FRUITSANDVEGETABLES", "POTATOES", "ONION", "FODDER"
]

# 2. Build a comprehensive lookup map for variations found in raw Indian datasets
CROP_MAPPING_DICTIONARY = {
    # Grains & Cereals
    "paddy": "RICE", "rice": "RICE",
    "wheat": "WHEAT",
    "jowar": "SORGHUM", "sorghum": "SORGHUM",
    "kharif jowar": "KHARIFSORGHUM", "kharif sorghum": "KHARIFSORGHUM",
    "rabi jowar": "RABISORGHUM", "rabi sorghum": "RABISORGHUM",
    "bajra": "PEARLMILLET", "pearl millet": "PEARLMILLET",
    "maize": "MAIZE", "corn": "MAIZE",
    "ragi": "FINGERMILLET", "finger millet": "FINGERMILLET",
    "barley": "BARLEY",
    
    # Pulses
    "gram": "CHICKPEA", "chickpea": "CHICKPEA", "chana": "CHICKPEA",
    "tur": "PIGEONPEA", "arhar": "PIGEONPEA", "pigeonpea": "PIGEONPEA",
    "moong": "MINORPULSES", "urad": "MINORPULSES", "lentil": "MINORPULSES", 
    "masur": "MINORPULSES", "pulses": "MINORPULSES", "minor pulses": "MINORPULSES",
    
    # Oilseeds
    "groundnut": "GROUNDNUT", "peanuts": "GROUNDNUT",
    "til": "SESAMUM", "sesamum": "SESAMUM", "sesame": "SESAMUM",
    "mustard": "RAPESEEDANDMUSTARD", "rapeseed": "RAPESEEDANDMUSTARD", "toria": "RAPESEEDANDMUSTARD",
    "safflower": "SAFFLOWER",
    "castor": "CASTOR",
    "linseed": "LINSEED",
    "sunflower": "SUNFLOWER",
    "soyabean": "SOYABEAN", "soybean": "SOYABEAN",
    "oilseeds": "OILSEEDS", "other oilseeds": "OILSEEDS",
    
    # Commercial & Horticulture
    "sugarcane": "SUGARCANE",
    "cotton": "COTTON",
    "fruits": "FRUITS",
    "vegetables": "VEGETABLES",
    "fruits & vegetables": "FRUITSANDVEGETABLES", "fruits and vegetables": "FRUITSANDVEGETABLES",
    "potato": "POTATOES", "potatoes": "POTATOES",
    "onion": "ONION",
    "fodder": "FODDER", "guar fodder": "FODDER"
}

def categorize_crop(crop_name):
    if pd.isna(crop_name):
        return np.nan
    
    # Clean the input string to match keys
    clean_name = str(crop_name).lower().strip().replace("_", " ").replace("-", " ")
    
    # Check 1: Direct translation from mapping dictionary
    if clean_name in CROP_MAPPING_DICTIONARY:
        return CROP_MAPPING_DICTIONARY[clean_name]
    
    # Check 2: See if alphanumeric stripping matches one of the selector categories directly
    alphanumeric_clean = "".join([c for c in clean_name if c.isalnum()]).upper()
    if alphanumeric_clean in SELECTOR_CROPS:
        return alphanumeric_clean
        
    return "UNKNOWN"

# Apply mapping rules to create your unified category anchor column
df_clean['Selector_Crop_Category'] = df_clean['Crop'].apply(categorize_crop)

# Print diagnostic breakdown of the categorization result
unknown_rows = df_clean[df_clean['Selector_Crop_Category'] == 'UNKNOWN']
print(f"Successfully categorized rows: {len(df_clean) - len(unknown_rows)} / {len(df_clean)}")
print(f"Rows left as UNKNOWN due to missing mapping keys: {len(unknown_rows)}")

if len(unknown_rows) > 0:
    print("\nSample of Unmapped Crop Strings (Add these to your map dictionary keys above):")
    print(unknown_rows['Crop'].unique()[:10])

Successfully categorized rows: 3395 / 4949
Rows left as UNKNOWN due to missing mapping keys: 1554

Sample of Unmapped Crop Strings (Add these to your map dictionary keys above):
<StringArray>
[    'Peas',  'Khesari',   'Kulthi',     'TIL/',     'Till',     'Jute',
    'Mesta',   'Kudrum',     'Til/', 'Lathyrus']
Length: 10, dtype: str


In [6]:
# Drop any rows that couldn't be matched to your application features
df_final = df_clean[df_clean['Selector_Crop_Category'] != 'UNKNOWN'].copy()

print(f"Final Sanatized Dataframe Shape: {df_final.shape}")

# Save directly to your Next.js public/data asset structure
df_final.to_csv("clean_crop_calendar_with_categories.csv", index=False)
print("File successfully prepared for API deployment wrapper integration!")

Final Sanatized Dataframe Shape: (3395, 18)
File successfully prepared for API deployment wrapper integration!
